In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Change only these if your Unity Catalog names are different
CATALOG = f"formula1_{env}"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

def bronze_table(name):
    return f"{CATALOG}.{BRONZE}.{name}"

def silver_table(name):
    return f"{CATALOG}.{SILVER}.{name}"

In [0]:
lap_times=spark.table(bronze_table("lap_times"))
lap_times.printSchema()
print("Bronze rows:",lap_times.count())

## 1. NULL + duplicate checks

In [0]:
display(lap_times.filter(
    F.col("race_id").isNull() |
    F.col("driver_id").isNull() |
    F.col("lap").isNull()
))
display(lap_times.groupBy("race_id","driver_id","lap")
                 .count().filter(F.col("count")>1))

## 2. String operations

In [0]:
lap_work=(
    lap_times
    .withColumn("time_clean",F.trim("time"))
    .withColumn("time_parts",F.split("time",":"))
    .withColumn("time_length",F.length("time_clean"))
    .withColumn("time_prefix",F.substring("time_clean",1,3))
)
display(lap_work.select(
    "race_id","driver_id","lap","time","time_clean",
    "time_parts","time_length","time_prefix"
).limit(20))

## 3. contains / startsWith / endsWith

In [0]:
display(lap_work.filter(F.col("time_clean").contains(":"))
                  .select("race_id","driver_id","time_clean").limit(20))
display(lap_work.filter(F.col("time_clean").startswith("1"))
                  .select("race_id","driver_id","time_clean").limit(20))
display(lap_work.filter(F.col("time_clean").endswith("0"))
                  .select("race_id","driver_id","time_clean").limit(20))

## 4. Filter + cast + CASE

In [0]:
lap_work=(
    lap_work
    .filter(F.col("race_id").isNotNull())
    .filter(F.col("driver_id").isNotNull())
    .filter(F.col("lap").isNotNull())
    .withColumn("milliseconds_clean",F.col("milliseconds").cast("long"))
    .withColumn(
        "lap_position_category",
        F.when(F.col("position")==1,"Fastest")
         .when(F.col("position").between(2,10),"Top 10")
         .otherwise("Other")
    )
)
display(lap_work.select(
    "race_id","driver_id","lap","position",
    "milliseconds_clean","lap_position_category"
).limit(20))

## 5. Incremental batch — `ingestion_date` is the watermark

In [0]:
target_name=silver_table("lap_times")

if not spark.catalog.tableExists(target_name):
    lap_times_batch=lap_times
else:
    last_ts=spark.table(target_name).agg(
        F.max("ingestion_date").alias("max_ts")
    ).first()["max_ts"]

    print("Last Silver ingestion_date:",last_ts)

    lap_times_batch=(
        lap_times if last_ts is None
        else lap_times.filter(F.col("ingestion_date")>F.lit(last_ts))
    )

print("Rows selected:",lap_times_batch.count())

## 6. Clean selected batch

In [0]:
lap_times_clean=(
    lap_times_batch
    .filter(F.col("race_id").isNotNull())
    .filter(F.col("driver_id").isNotNull())
    .filter(F.col("lap").isNotNull())
    .dropDuplicates(["race_id","driver_id","lap"])
    .withColumn("time_clean",F.trim("time"))
    .withColumn("milliseconds_clean",F.col("milliseconds").cast("long"))
    .withColumn(
        "lap_position_category",
        F.when(F.col("position")==1,"Fastest")
         .when(F.col("position").between(2,10),"Top 10")
         .otherwise("Other")
    )
    .withColumn("silver_processed_timestamp",F.current_timestamp())
)
display(lap_times_clean.limit(20))

In [0]:
lap_times_clean.count()

## 7. MERGE

In [0]:
if not spark.catalog.tableExists(target_name):
    (lap_times_clean.write.format("delta").mode("overwrite")
     .option("overwriteSchema","true").saveAsTable(target_name))
else:
    target=DeltaTable.forName(spark,target_name)
    (target.alias("t")
     .merge(
         lap_times_clean.alias("s"),
         "t.race_id=s.race_id AND t.driver_id=s.driver_id AND t.lap=s.lap"
     )
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())

print("LAP_TIMES MERGE completed.")

In [0]:
silver=spark.table(target_name)
print("Silver rows:",silver.count())
display(silver.orderBy(F.desc("ingestion_date")).limit(20))